# 4. Conclusiones, Limitaciones y Reproducibilidad

## 4.1 Conclusiones

El proyecto se formuló como un problema de **regresión temporal** para predecir `target_return_t1`, el rendimiento logarítmico de la siguiente hora, utilizando datos horarios de BTCUSDT, ETHUSDT, BNBUSDT, XRPUSDT y SOLUSDT de Binance Spot.

La separación entre entrenamiento y prueba se realizó cronológicamente antes del modelado. Los rezagos y ventanas móviles se construyeron únicamente con información pasada, y el preprocesamiento se integró dentro de un `Pipeline` con validación temporal.

El modelo base supervisado fue `LinearSVR`, comparado contra una línea base de persistencia. En el conjunto de prueba se obtuvo:

| Modelo | RMSE | $R^2$ |
|---|---:|---:|
| Persistencia | 0.874818 | -0.991587 |
| LinearSVR | 0.620891 | -0.003215 |

El `LinearSVR` redujo el RMSE aproximadamente un **29.03 %** respecto a persistencia. Sin embargo, el valor $R^2=-0.003215$ indica una capacidad explicativa prácticamente nula sobre la variabilidad del rendimiento horario futuro.

La curva de aprendizaje mostró que el RMSE de validación disminuyó de 0.858093 con el 20 % del entrenamiento a 0.838704 con el 100 %, pero la mejora se volvió marginal a partir de aproximadamente el 60 % de los datos. Esto sugiere que el tamaño muestral es suficiente para estabilizar el modelo base y que sus limitaciones no se explican principalmente por falta de observaciones.

Las medias de los residuos fueron cercanas a cero para los cinco activos. Jarque--Bera produjo valores $p<0.001$ en todos los casos, por lo que se rechaza la normalidad. Breusch--Pagan también produjo valores $p<0.001$, evidenciando heterocedasticidad.

La ACF de los residuos fue pequeña. En el rezago 1 los valores fueron 0.027260 para BNBUSDT, 0.002248 para BTCUSDT, 0.034047 para ETHUSDT, 0.025130 para SOLUSDT y 0.017020 para XRPUSDT. Los rezagos 2, 3, 6, 12, 24 y 48 también permanecieron próximos a cero.

En conjunto, el modelo supera a la persistencia en RMSE, pero la señal lineal disponible para explicar el siguiente rendimiento horario es muy limitada. La estructura restante parece asociarse principalmente con heterocedasticidad, colas pesadas, eventos extremos y posibles relaciones no lineales.

## 4.2 Limitaciones

Los resultados corresponden exclusivamente a **Binance Spot** y a cinco criptoactivos, por lo que no representan necesariamente todo el mercado de criptomonedas.

La frecuencia analizada es horaria. Los resultados podrían cambiar con horizontes de minutos, cuatro horas o datos diarios.

El conjunto de predictores se basa principalmente en OHLCV, actividad de mercado, rezagos, ventanas móviles y variables temporales. No se incorporaron noticias, sentimiento, variables macroeconómicas, datos on-chain, libro de órdenes, derivados, funding rates u open interest.

`LinearSVR` es un modelo lineal y puede resultar insuficiente para capturar interacciones no lineales, cambios de régimen, clustering de volatilidad y comportamientos extremos.

MAPE se reporta por requisito metodológico, pero debe interpretarse con cautela porque los rendimientos pueden ser negativos o encontrarse muy cerca de cero.

La heterocedasticidad detectada indica que la varianza del error no es constante y motiva considerar posteriormente modelos capaces de representar explícitamente la volatilidad condicional.

Las Secciones 2.7 y 2.8 no aplican porque el dataset no contiene latitud, longitud ni unidades geográficas. El problema pertenece a la **Ruta C: Regresión temporal**.

## 4.3 Reproducibilidad

La semilla aleatoria utilizada es `SEED = 42`.

La división `train/test` es cronológica y el conjunto de prueba permaneció reservado hasta la evaluación final. La validación cruzada se realizó mediante `TimeSeriesSplit` sobre timestamps únicos, manteniendo juntos los cinco activos correspondientes a una misma hora y utilizando un `gap` de 24 horas.

Los rezagos se construyeron mediante `shift(k)` con $k\geq1$, y las ventanas móviles utilizaron `shift(1).rolling(...)`. Los cálculos respetaron `symbol` y los segmentos temporales continuos para evitar atravesar discontinuidades.

El preprocesamiento se implementó mediante `Pipeline` y `ColumnTransformer`. La imputación residual, estandarización y codificación categórica se ajustaron únicamente con cada bloque de entrenamiento.

Las dependencias del proyecto están documentadas en `requirements.txt`. El archivo `model_evaluation.ipynb` contiene el EDA, el preprocesamiento, el modelo base y esta sección final, manteniendo en un único notebook la trazabilidad entre código, resultados e interpretación.

## 4.4 Conclusión final

El `LinearSVR` obtuvo un RMSE de **0.620891**, frente a **0.874818** para persistencia, lo que representa una mejora aproximada del **29.03 %**. No obstante, $R^2=-0.003215$ muestra que la capacidad explicativa lineal del rendimiento de la siguiente hora continúa siendo prácticamente nula.

Los residuos presentan medias próximas a cero, no normalidad, heterocedasticidad y autocorrelación lineal de pequeña magnitud. La curva de aprendizaje también indica que aumentar el tamaño de entrenamiento produce mejoras cada vez menores.

Por tanto, `LinearSVR` cumple adecuadamente su función como **modelo base**. Cualquier modelo posterior deberá evaluarse bajo la misma separación cronológica y demostrar una mejora fuera de muestra frente tanto a `LinearSVR` como a la línea base de persistencia.